# 004 · Scope and Closures

> **Syntax › Functions** · Python 3.14

Python's scoping rules are simpler than C#'s, but different in ways that regularly surprise C# developers: there is **no block scope**, and assigning to a variable anywhere in a function makes it local to that whole function.

**In this notebook you will learn:**

- the LEGB rule Python uses to look up names
- why there is no block scope, and what that means for `if`/`for`/`with` blocks
- `global` and `nonlocal`, and when (rarely) to use them
- the `UnboundLocalError` gotcha
- closures, and how they compare to C# lambdas capturing variables
- the late-binding closure gotcha in loops, and its fixes

## C# mental model

| C# | Python |
|---|---|
| block scope: `{ ... }` creates a scope | only **functions** (and classes, modules, comprehensions) create scopes |
| variable declared with `var x` | variable created by the first assignment |
| static field for "global" state | module-level variable |
| lambda/local function captures a variable | closure captures a variable (same idea!) |
| captured variables can be reassigned | captured variables can only be **reassigned** if declared `nonlocal` |
| `foreach` gives a fresh variable each iteration (C# 5+) | `for` reuses **one** variable for the whole loop |

## The LEGB rule

When Python looks up a name, it searches four scopes in order and uses the first match:

1. **L**ocal: names assigned inside the current function
2. **E**nclosing: locals of any enclosing functions (for nested functions)
3. **G**lobal: names at the top level of the module (the notebook, here)
4. **B**uilt-in: names like `len`, `print`, `range`

In [ ]:
x = "global"


def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner sees:", x)

    inner()
    print("outer sees:", x)


outer()
print("module sees:", x)

Remove the inner assignments and the lookup falls through to the next scope out:

In [ ]:
x = "global"


def outer():
    def inner():
        print("inner sees:", x, "| and len is", len)   # global, then built-in
    inner()


outer()

⚠️ **Gotcha:** built-ins are just the last scope in the chain, so you can accidentally **shadow** them. Avoid naming variables `list`, `dict`, `id`, `type`, `sum`, `input`, ...

In [ ]:
def broken():
    list = [1, 2, 3]          # shadows the built-in list() inside this function
    try:
        return list("abc")
    except TypeError as e:
        return f"{type(e).__name__}: {e}"


print(broken())

## No block scope

This is the biggest difference from C#. `if`, `for`, `while`, `with` and `try` blocks do **not** create a scope. A variable assigned inside them is visible after the block ends.

```csharp
// C#
if (true)
{
    var message = "inside";
}
// Console.WriteLine(message);   // CS0103: name does not exist in the current context

for (var i = 0; i < 3; i++) { }
// Console.WriteLine(i);         // also an error
```

In [ ]:
if True:
    message = "inside"
print(message)            # works: no block scope

for i in range(3):
    last_square = i * i
print(i, last_square)     # the loop variable survives the loop: 2 4

⚠️ **Gotcha:** the flip side is that a variable may simply **not exist** if the block never ran. C#'s definite-assignment analysis would catch this at compile time; Python fails at runtime.

In [ ]:
def find_first_negative(numbers):
    for n in numbers:
        if n < 0:
            found = n
            break
    return found        # only assigned if the loop found something


print(find_first_negative([3, -1, 4]))
try:
    find_first_negative([1, 2, 3])
except UnboundLocalError as e:
    print(f"{type(e).__name__}: {e}")

The fix is to initialise before the block (`found = None`). Note that **comprehensions do have their own scope**, so their loop variable does not leak *(see Collections › 007 · Comprehensions)*:

In [ ]:
squares = [k * k for k in range(3)]
try:
    print(k)
except NameError as e:
    print(f"{type(e).__name__}: {e}")

## Assignment makes a name local

Python decides at **compile time** whether a name is local to a function: if the function assigns to the name *anywhere* (including `+=`), it is local *everywhere* in that function. Reading a global is fine; assigning to it creates a new local instead.

In [ ]:
counter = 0


def read_only():
    return counter + 1    # just reads the global: fine


def shadows():
    counter = 100         # creates a NEW local; the global is untouched
    return counter


print(read_only(), shadows(), counter)

## ⚠️ Gotcha: `UnboundLocalError`

Combine "reading" and "assigning" in the same function and you get a classic error. Because of the assignment, `counter` is local for the **whole** function, so the read happens before the local has a value.

In [ ]:
counter = 0


def increment():
    counter += 1          # read + assign -> counter is local everywhere here
    return counter


try:
    increment()
except UnboundLocalError as e:
    print(f"{type(e).__name__}: {e}")

## `global` and `nonlocal`

To **rebind** a name from an outer scope, you must declare it:

- `global name` rebinds a module-level variable
- `nonlocal name` rebinds a variable in the nearest enclosing function

Mutating an object (e.g. `items.append(...)`) needs no declaration, because that is not rebinding the name.

In [ ]:
counter = 0


def increment():
    global counter
    counter += 1
    return counter


increment()
increment()
print(counter)   # 2

`global` is usually a design smell, just like mutable static state in C#. Prefer returning values, or a class *(see Classes & OOP › 001 · Classes and Objects)*. `nonlocal` is more common, because it is how closures keep mutable state:

In [ ]:
def outer():
    count = 0

    def bump():
        nonlocal count     # without this: UnboundLocalError
        count += 1

    bump()
    bump()
    return count


print(outer())   # 2

## Closures

A **closure** is a function that remembers variables from the scope where it was created, even after that scope has finished. C# lambdas and local functions do exactly the same thing: the compiler hoists captured variables into a hidden class.

```csharp
// C#
static Func<int> MakeCounter()
{
    var count = 0;
    return () => ++count;      // captures 'count' by reference
}

var next = MakeCounter();
next(); next();                // 1, 2
```

In [ ]:
def make_counter():
    count = 0

    def next_value():
        nonlocal count
        count += 1
        return count

    return next_value


counter_a = make_counter()
counter_b = make_counter()          # independent state
print(counter_a(), counter_a(), counter_a())
print(counter_b())

Like C#, Python captures **variables, not values**: the closure sees the variable's *current* value at the time it runs. You can even inspect the captured cells:

In [ ]:
def make_greeter(greeting):
    def greet(name):
        return f"{greeting}, {name}!"
    return greet


hello = make_greeter("Hello")
print(hello("alice"))
print(hello.__code__.co_freevars, hello.__closure__[0].cell_contents)

## ⚠️ Gotcha: late-binding closures in loops

In C# 5+, `foreach` creates a **fresh variable per iteration**, so each lambda captures its own value. (The classic `for (int i...)` loop still shares one variable.) A Python `for` loop reuses **one** variable, so every closure created in the loop sees its **final** value.

```csharp
// C#
var actions = new List<Func<int>>();
foreach (var i in Enumerable.Range(0, 3))
    actions.Add(() => i);
// foreach: 0, 1, 2      (fresh variable per iteration)
// for (int i = 0; ...): 3, 3, 3   (one shared variable)
```

In [ ]:
funcs = []
for i in range(3):
    funcs.append(lambda: i)          # captures the variable i, not its value

print([f() for f in funcs])          # [2, 2, 2], not [0, 1, 2]

The same happens with comprehensions, and with `def` inside a loop. There are two common fixes:

1. **Default argument**: defaults are evaluated when the function is *created*, so `i=i` snapshots the current value.
2. **`functools.partial`**: binds the value as an argument up front *(see Functions › 005 · Lambdas and Higher-Order Functions)*.

In [ ]:
from functools import partial

# Fix 1: default argument captures the value at definition time
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])


# Fix 2: partial binds the current value
def identity(value):
    return value


funcs = [partial(identity, i) for i in range(3)]
print([f() for f in funcs])

A third option is a factory function: each call to `make_greeter` (above) creates a new scope, so each closure gets its own variable.

In [ ]:
def make_multiplier(factor):
    return lambda x: x * factor     # 'factor' is a fresh variable per call


multipliers = [make_multiplier(n) for n in range(1, 4)]
print([m(10) for m in multipliers])

## Summary

| Concept | C# | Python |
|---|---|---|
| What creates a scope | any `{ }` block | functions, classes, modules, comprehensions |
| Variable visible after `if`/`for` | no | yes |
| Name lookup | compiler-resolved | LEGB: Local → Enclosing → Global → Built-in |
| Assign to outer variable | just assign | `nonlocal x` / `global x` |
| Read-before-assign in function | compile error | `UnboundLocalError` at runtime |
| Closure | lambda / local function captures variables | nested function captures variables |
| Loop capture | `foreach`: fresh per iteration | one shared variable: use `x=x` default or `partial` |

## Exercises

1. Predict the output, then run it: `x = 1`, then `def f(): print(x); x = 2`, then call `f()`. Explain the result.
2. Write `make_accumulator()` returning a function that adds its argument to a running total and returns the new total, e.g. `acc(10) -> 10`, `acc(5) -> 15`.
3. Fix this so it prints `0 1 2`: `handlers = [lambda: print(n) for n in range(3)]` followed by `for h in handlers: h()`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — UnboundLocalError: the assignment makes x local for the whole of f,
#     so print(x) reads the local before it has a value.

# 2 — nonlocal lets the closure rebind the enclosing variable
def make_accumulator():
    total = 0
    def add(amount):
        nonlocal total
        total += amount
        return total
    return add

acc = make_accumulator()
print(acc(10), acc(5))   # 10 15

# 3 — snapshot n with a default argument
handlers = [lambda n=n: print(n) for n in range(3)]
for h in handlers:
    h()
```

</details>